# 🚀 Спасти марсоход

Практическое занятие по Python: Google Colab → Jupyter Notebook → Git → Docker.

**180 минут с перерывом · работа в парах · начальный уровень.** Нужно знать переменные, условия и циклы. Если это первый опыт программирования, разделите материал на два занятия: этапы 1–4 и этапы 5–8.

Марсоходу нужно добраться до станции. У маршрутов разный расход энергии и опасность. Ваша задача — выбрать допустимый маршрут, проверить алгоритм и передать работающую программу следующей смене. Все числа — условные параметры учебной симуляции.

**Результат:** блокнот с экспериментами, программа `mission.py`, история Git и образ Docker.

**ВЕРСИЯ ПРЕПОДАВАТЕЛЯ: решение включено.**


## 1. Подготовка и роли · 10 минут

В паре один участник пишет код, второй предсказывает результат и придумывает проверки. Меняйтесь каждые 20–25 минут.

- В Colab: откройте этот `.ipynb` через «Файл → Загрузить блокнот». Интернет и доступ к Google нужны заранее. GPU не требуется.
- В Jupyter: из папки проекта запустите `jupyter notebook`, затем откройте блокнот.
- Git и Docker используйте **в локальном терминале**, не в ячейках Colab.
- GitHub показывает блокнот, но не исполняет Python в обычном просмотре файла.

**Критерий успеха:** вы объясняете, где выполняется код и где хранится файл.


## 2. Первый контакт · 15 минут

Запустите ячейку через Shift+Enter. Измените имя и заряд. Сначала предскажите вывод, затем запустите.

Вопрос: изменится ли значение переменной в памяти, если отредактировать ячейку, но не выполнить её?


In [ ]:
rover_name = "Искатель"
battery = 60
print(f"Марсоход {rover_name} готов к работе")
print(f"Заряд: {battery} единиц")


## 3. Выбор маршрута · 30 минут

| Маршрут | Энергия | Опасность |
|---|---:|---:|
| Кратер | 35 | 7 |
| Равнина | 50 | 2 |
| Каньон | 45 | 4 |

Правила: расход не превышает заряд; опасность не превышает лимит; среди допустимых выбираем минимальный расход. При равенстве оставляем первый маршрут. Если подходящего маршрута нет — `None`.

При заряде 60 и лимите опасности 4 сначала выберите маршрут вручную.

**Задание:** заполните `choose_route` в следующей ячейке. Остальной код уже готов. На этом этапе `reserve=0`; резерв добавим в финале.

Первая строка `%%writefile mission.py` записывает содержимое ячейки в файл текущей среды. После каждой правки запускайте ячейку заново. Она **перезаписывает** `mission.py`: не храните в нём отдельные несохранённые правки.

Подсказки: заведите `best_route = None`; переберите список; соедините ограничения через `and`; обновляйте лучший маршрут, только если новый расход меньше. До реализации функции `NotImplementedError` — ожидаемый результат заготовки.


In [ ]:
%%writefile mission.py
ROUTES = [
    {"name": "Кратер", "energy": 35, "risk": 7},
    {"name": "Равнина", "energy": 50, "risk": 2},
    {"name": "Каньон", "energy": 45, "risk": 4},
]

def choose_route(routes, battery, max_risk, reserve=0):
    """Выбрать самый экономичный допустимый маршрут; при равенстве — первый."""
    if battery < 0 or max_risk < 0 or reserve < 0:
        raise ValueError("Параметры миссии не могут быть отрицательными")
    best_route = None
    for route in routes:
        enough_energy = route["energy"] + reserve <= battery
        safe_enough = route["risk"] <= max_risk
        if enough_energy and safe_enough:
            if best_route is None or route["energy"] < best_route["energy"]:
                best_route = route
    return best_route

def main():
    import argparse
    parser = argparse.ArgumentParser(description="Миссия марсохода")
    parser.add_argument("--battery", type=int, default=60)
    parser.add_argument("--max-risk", type=int, default=4)
    parser.add_argument("--reserve", type=int, default=0)
    args = parser.parse_args()
    if min(args.battery, args.max_risk, args.reserve) < 0:
        parser.error("Параметры миссии не могут быть отрицательными")
    selected = choose_route(ROUTES, args.battery, args.max_risk, args.reserve)
    if selected is None:
        print("Безопасного маршрута нет. Остаёмся на месте.")
    else:
        print("Маршрут:", selected["name"])
        print("Остаток энергии:", args.battery - selected["energy"])

if __name__ == "__main__":
    main()


Запишите файл предыдущей ячейкой, затем выполните эту. `runpy` читает актуальный файл при каждом запуске.

In [ ]:
import runpy
mission = runpy.run_path("mission.py")
selected = mission["choose_route"](mission["ROUTES"], 60, 4)
print(selected)


## 4. Испытания и «саботаж» · 20 минут

Предскажите результаты, затем запустите проверки.

| Заряд | Лимит опасности | Ожидается |
|---:|---:|---|
| 60 | 4 | Каньон |
| 60 | 2 | Равнина |
| 40 | 4 | Нет маршрута |
| 45 | 4 | Каньон |
| 35 | 7 | Кратер |

После успешной проверки временно замените `<=` на `<` в ограничении энергии. Снова запишите файл и запустите проверки. Какой случай обнаружил ошибку? Исправьте её.

**Затем перерыв 10 минут.**


In [ ]:
def check_mission(choose_route, final=False):
    routes = [
        {"name": "Кратер", "energy": 35, "risk": 7},
        {"name": "Равнина", "energy": 50, "risk": 2},
        {"name": "Каньон", "energy": 45, "risk": 4},
    ]
    cases = [(60, 4, "Каньон"), (60, 2, "Равнина"),
             (40, 4, None), (45, 4, "Каньон"), (35, 7, "Кратер")]
    count = 0
    for battery, risk, expected in cases:
        result = choose_route(routes, battery, risk)
        name = None if result is None else result["name"]
        if name != expected:
            raise AssertionError(f"Заряд={battery}, риск={risk}: ожидалось {expected}, получено {name}")
        count += 1
    if choose_route([], 60, 4) is not None:
        raise AssertionError("Для пустого списка ожидается None")
    count += 1
    tied = [{"name": "Первый", "energy": 20, "risk": 1},
            {"name": "Второй", "energy": 20, "risk": 1}]
    if choose_route(tied, 20, 1)["name"] != "Первый":
        raise AssertionError("При равной энергии оставьте первый маршрут")
    count += 1
    for params in [(-1, 4, 0), (60, -1, 0), (60, 4, -1)]:
        try:
            choose_route(routes, *params)
        except ValueError:
            count += 1
        else:
            raise AssertionError("Отрицательные параметры должны вызывать ValueError")
    if final:
        for battery, reserve, expected in [(55, 10, "Каньон"), (54, 10, None), (60, 61, None)]:
            result = choose_route(routes, battery, 4, reserve)
            name = None if result is None else result["name"]
            if name != expected:
                raise AssertionError(f"Заряд={battery}, резерв={reserve}: ожидалось {expected}, получено {name}")
            count += 1
    print(f"Проверки пройдены: {count}")


In [ ]:
mission = runpy.run_path("mission.py")
check_mission(mission["choose_route"])


## 5. Переезд в Jupyter Notebook · 20 минут

1. Скачайте заполненный блокнот из Colab: «Файл → Скачать → Скачать .ipynb».
2. Положите его в локальную папку `mars-mission` из архива. Не открывайте одновременно исходный и заполненный блокноты для редактирования одного `mission.py`.
3. В терминале этой папки запустите `jupyter notebook` и откройте заполненный файл.
4. Перезапустите ядро и выполните ячейки сверху вниз. Ячейка `%%writefile` создаст локальный `mission.py` из вашего решения.
5. В **терминале** запустите:

```bash
python mission.py --battery 60 --max-risk 4
python check.py mission.py
```

На системах, где команда называется `python3`, замените `python` на `python3`.

Ожидается маршрут «Каньон», остаток 15 и 10 пройденных проверок. Файл в Colab и локальный файл — разные копии. Файлы временной среды Colab следует скачать до завершения сессии.

С этого момента редактируйте локальный `mission.py`. Если снова запустите старую ячейку `%%writefile`, она затрёт ваши новые правки: сначала перенесите их обратно в блокнот.


## 6. Git — журнал экспедиции · 25 минут

Все команды — в терминале папки проекта. Если папка получена через `git clone`, пропустите `git init`.

```bash
git init
git config user.name "Mars Team 1"
git config user.email "mars-team-1@example.com"
git status
git add mission.py mission.ipynb check.py .gitignore
git commit -m "Реализован выбор маршрута"
```

Это учебные подписи коммитов, настройки действуют только в этом репозитории. Git работает локально без аккаунта GitHub. Если заполненный блокнот называется иначе, подставьте его имя вместо `mission.ipynb`.

**Эксперимент:** добавьте в список `ROUTES` в `mission.py` маршрут:

```python
{"name": "Лавовая труба", "energy": 30, "risk": 3},
```

При заряде 60 и лимите 4 программа должна выбрать его. Затем:

```bash
python mission.py
git diff
git add mission.py
git commit -m "Добавлен маршрут через лавовую трубу"
git log --oneline
```

`check.py` проверяет функцию на собственном фиксированном наборе маршрутов, поэтому добавление нового маршрута в `ROUTES` не меняет ожидаемые результаты этих проверок.

Обсудите: что делает `add`, чем коммит отличается от сохранения файла, почему коммит ещё не означает отправку на GitHub?


## 7. Docker — передача следующей смене · 25 минут

Откройте `Dockerfile` в корне проекта:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY mission.py .
ENTRYPOINT ["python", "mission.py"]
CMD ["--battery", "60", "--max-risk", "4"]
```

`FROM` выбирает основу с Python, `WORKDIR` — папку внутри контейнера, `COPY` копирует файл, `ENTRYPOINT` запускает программу, `CMD` задаёт аргументы по умолчанию.

В локальном терминале из корня проекта:

```bash
docker build -t mars-mission .
docker run --rm mars-mission
docker run --rm mars-mission --battery 25 --max-risk 4
```

Точка в команде сборки обязательна. Второй запуск должен сообщить об отсутствии подходящего маршрута. `--rm` удаляет завершившийся контейнер, но сохраняет образ.

Измените сообщение в `mission.py`, запустите старый образ, затем пересоберите и запустите снова. Объясните, почему новая надпись появилась лишь после сборки.

```bash
git add Dockerfile .dockerignore mission.py
git commit -m "Добавлен запуск через Docker"
```

Если Docker недоступен на вашем ПК, передайте `mission.py` преподавателю. Он соберёт и запустит проект на своём компьютере, а вы предскажете результат. Это демонстрация, а не индивидуальная практика Docker.


## 8. Финальное испытание: пыльная буря · 20 минут

После поездки должно остаться минимум 10 единиц энергии для связи.

- Добавьте учёт `reserve` в функцию в локальном `mission.py`.
- Интерфейс `--reserve` уже подготовлен. Менять разбор аргументов не требуется.
- При исходных трёх маршрутах заряд 55 и резерв 10 разрешают «Каньон», а заряд 54 — уже нет. Добавленная «Лавовая труба» меняет результат реального запуска; автоматические тесты используют исходный список.

```bash
python check.py mission.py --final
python mission.py --battery 55 --max-risk 4 --reserve 10
docker build -t mars-mission .
docker run --rm mars-mission --battery 55 --max-risk 4 --reserve 10
git add mission.py
git commit -m "Учтён резерв энергии для связи"
```

Ожидаются **13 пройденных проверок**. Обновите ячейку с программой в блокноте из финального `mission.py`, выполните её и проверки, сохраните блокнот и закоммитьте его. Теперь версии снова совпадают.

Дополнительно для быстрых команд: при равной энергии выбирать меньшую опасность; загрузить маршруты из JSON; добавить текстовую диаграмму расходов. При изменении правила равенства обновите соответствующий тест и объясните почему.


После переноса финальной версии обратно в ячейку `%%writefile` выполните её, затем проверку ниже.

In [ ]:
mission = runpy.run_path("mission.py")
check_mission(mission["choose_route"], final=True)


## 9. Защита и оценка · 5 минут

Покажите запуск, историю Git и один тест, который обнаруживал ошибку.

| Критерий | Баллы |
|---|---:|
| Правильный выбор маршрута | 2 |
| Обработка отсутствия маршрута | 1 |
| Проверка точного равенства энергии | 2 |
| Блокнот работает после перезапуска ядра | 1 |
| Два осмысленных коммита | 1 |
| Самостоятельный запуск Docker | 1 |
| Резерв энергии и его проверки | 2 |
| Итого | 10 |

При демонстрационном Docker оцените этот пункт по объяснению результата и содержимого Dockerfile, отметив формат работы.

Ответьте: где выполнялся Python в Colab? Что хранит `.ipynb`? Что сохраняет коммит? Чем образ отличается от контейнера? Почему после правки нужна сборка?
